# Solution E1 · The model that finds signal in noise

**Type**: Find the bug

## Objective

A colleague evaluated a model on pure noise data and got a positive cross-validated R². Find
the mistake, fix it with a `Pipeline`, and measure by how much the evaluation was optimistic.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

cv = KFold(5, shuffle=True, random_state=0)

## Commented solution

**Where the bug is.** `SelectKBest(...).fit_transform(X, y)` is run on the *whole* dataset, before the
cross-validation. To choose the 10 features with the highest correlation with `y`, it looked at the `y` of all the
observations, including those that then end up in the validation folds. Among 1000 noise columns, the 10 best
have, by pure luck, a correlation with `y` of about 0.3, and the subsequent cross-validation "confirms" it because
those correlations also hold on the validation folds: it already used them to be chosen.

In [ ]:
def experiment(seed):
    X, y = ds.pure_noise(seed=seed)

    # --- the colleague's code: the selection sees ALL of y ---
    X_sel = SelectKBest(f_regression, k=10).fit_transform(X, y)
    r2_bug = cross_val_score(LinearRegression(), X_sel, y, cv=cv, scoring="r2").mean()

    # --- fixed: the selection is part of the pipeline, so it is redone on every training fold ---
    pipe = Pipeline([("sel", SelectKBest(f_regression, k=10)), ("lr", LinearRegression())])
    r2_ok = cross_val_score(pipe, X, y, cv=cv, scoring="r2").mean()
    return r2_bug, r2_ok


r2_bug_0, r2_ok_0 = experiment(0)
print(f"colleague's R2: {r2_bug_0:.3f} | fixed R2: {r2_ok_0:.3f}")

In [ ]:
results = np.array([experiment(s) for s in range(10)])
mean_optimism = np.mean(results[:, 0] - results[:, 1])
print("colleague's R2 per seed:", results[:, 0].round(2))
print("fixed R2 per seed      :", results[:, 1].round(2))
print(f"mean optimism: {mean_optimism:.2f} points of R2")
answer = "b"

## Check

In [ ]:
with verify():
    check(r2_bug_0 > 0.15, "the colleague's number is positive and looks like a good model",
          "r2_bug should be about 0.26 on this dataset: don't modify the colleague's code")
    check(r2_ok_0 < 0, "the fixed R2 is negative: in noise there is nothing to learn",
          "r2_ok must come from a Pipeline with SelectKBest INSIDE the cross-validation")
    check(results.shape == (10, 2) and bool(np.all(results[:, 0] > results[:, 1])),
          "in all 10 seeds the colleague's evaluation is more optimistic",
          "results must have 10 rows and r2_bug > r2_ok in every row")
    check(mean_optimism > 0.4, "the mean optimism is more than 0.4 points of R2",
          "mean_optimism should be the mean of r2_bug - r2_ok over 10 seeds")
    check(answer == "b", "correct explanation", "reread: what does a model learn from 10 random features on 80 observations?")
done()

## What you learned

- Any step that **learns something from the target** (feature selection, choice of hyperparameters, target-based
  imputations, target encoding) must sit inside the cross-validation. Outside, the evaluation is contaminated.
- The optimism here is huge: from an R² of about 0.2-0.3 (a "decent" model) to one of about -0.5 (worse than the
  mean) on data that contain nothing. The leakage grows when p is large compared to n, which is exactly the
  situation in which you do feature selection.
- A negative R² on data with no signal is not a bug: the model estimated 10 coefficients from 80 points and all of them are
  noise, so on new data it does worse than the mean.

## Common pitfall

Thinking that "doing the cross-validation" is enough. Cross-validation only protects from evaluating on data seen in
training: it does not protect from preprocessing and selection run beforehand. Rule of thumb: if a transformation has a
`fit`, it must sit in a `Pipeline`.

## Interview question

<details><summary>You have 5000 features and 200 patients. You selected the 20 most correlated with the outcome and then did the cross-validation, with AUC 0.85. Do you trust it?</summary>

No: the selection saw the labels of all the patients, including those of the validation folds, so the AUC is
inflated. With p much greater than n it is easy to find 20 features that correlate with the outcome by chance. I would redo the evaluation
with the selection inside a Pipeline (or a nested cross-validation), and compare with the AUC obtained by permuting
the labels: if that one is also high, the number means nothing.

</details>